### Code Documentation & Test Generator

This notebook builds a small **Gradio tool powered by an LLM** that can automatically:

- Add **docstrings and helpful comments** to code
- Generate **unit tests**

The interface allows switching between **Python** and **Ruby**, enabling the model to generate the appropriate test framework:

- **pytest** for Python  
- **RSpec** for Ruby

Run the cell below to launch the interactive interface.

In [1]:
# Load your imports
import os
from dotenv import load_dotenv
from textwrap import dedent
import gradio as gr
from openai import OpenAI

### Prepare LLM Clients
We will be using Deepseek, Gemini & OpenAI

In [25]:
# Load llm provider api keys
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

# urls
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"

# Initialize llm providers
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
openai = OpenAI()

providers = {
    "OpenAI": openai,
    "Gemini": gemini,
    "DeepSeek": deepseek,
}

PROVIDER_MODELS = {
    "OpenAI": [
        "gpt-4.1",
        "gpt-4.1-mini",
        "gpt-4o",
        "gpt-4o-mini"
    ],
    "Gemini": [
        "gemini-2.5-flash-lite",
        "gemini-2.5-flash",
        "gemini-2.5-pro"
    ],
    "DeepSeek": [
        "deepseek-chat",
        "deepseek-coder"
    ]
}

In [27]:
LANGUAGE_RULES = {
    "Python": {
        "commenting_instruction": dedent("""
            Add high-quality Python docstrings and concise inline comments only where useful.
            Prefer clean, professional Python style.
            Do not over-comment obvious lines.
            Use triple-quoted docstrings for modules, classes, and functions where appropriate.
        """).strip(),
        "test_framework": "pytest",
        "test_instruction": dedent("""
            Generate pytest unit tests.
            Use clear test names.
            Cover normal cases, edge cases, and error cases where reasonable.
            Return only the test code.
        """).strip(),
        "default_test_filename": "test_generated.py",
    },
    "Ruby": {
        "commenting_instruction": dedent("""
            Add high-quality Ruby comments and documentation where helpful.
            Prefer concise, idiomatic Ruby comments.
            Do not over-comment obvious lines.
            Add method/class comments only when they improve maintainability.
        """).strip(),
        "test_framework": "RSpec",
        "test_instruction": dedent("""
            Generate RSpec tests.
            Use clear example descriptions.
            Cover happy paths, edge cases, and failure scenarios where reasonable.
            Return only the spec code.
        """).strip(),
        "default_test_filename": "generated_spec.rb",
    },
}

TASK_RULES = {
    "Add docstrings/comments only": "annotate_only",
    "Generate unit tests only": "tests_only",
    "Do both": "both",
}


# -----------------------------------------------------------------------------
# LLM helpers
# -----------------------------------------------------------------------------

def build_messages(code: str, language: str, task: str):
    rules = LANGUAGE_RULES[language]
    task_mode = TASK_RULES[task]

    system_message = dedent(f"""
        You are a senior software engineer and code-quality assistant.

        Your job:
        1. Improve code readability and maintainability.
        2. Add comments/docstrings only when useful.
        3. Generate appropriate tests when requested.
        4. Respect the selected programming language.
        5. For Python, generate pytest tests.
        6. For Ruby, generate RSpec tests.

        General rules:
        - Preserve the original logic unless a tiny correction is absolutely necessary.
        - Do not invent missing dependencies unless needed for valid tests.
        - Keep comments/docstrings accurate and professional.
        - Do not wrap the final answer in markdown fences.
        - Return valid JSON only, with keys:
          annotated_code
          generated_tests
          notes
    """).strip()

    if task_mode == "annotate_only":
        user_message = dedent(f"""
            Language: {language}
            Task: Add docstrings/comments only

            Instructions:
            {rules["commenting_instruction"]}

            Return JSON with:
            - annotated_code: the improved code with docstrings/comments
            - generated_tests: empty string
            - notes: brief summary of what you added

            Input code:
            {code}
        """).strip()

    elif task_mode == "tests_only":
        user_message = dedent(f"""
            Language: {language}
            Task: Generate unit tests only

            Instructions:
            Framework: {rules["test_framework"]}
            {rules["test_instruction"]}

            Return JSON with:
            - annotated_code: empty string
            - generated_tests: test code only
            - notes: brief summary of test coverage

            Input code:
            {code}
        """).strip()

    else:  # both
        user_message = dedent(f"""
            Language: {language}
            Task: Add docstrings/comments and generate unit tests

            Commenting instructions:
            {rules["commenting_instruction"]}

            Test instructions:
            Framework: {rules["test_framework"]}
            {rules["test_instruction"]}

            Return JSON with:
            - annotated_code: improved code with docstrings/comments
            - generated_tests: test code only
            - notes: brief summary of what was added and what tests cover

            Input code:
            {code}
        """).strip()

    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]


def call_llm(code: str, language: str, task: str, provider: str, model: str):
    messages = build_messages(code, language, task)

    response = providers[provider].chat.completions.create(
        model=model,
        messages=messages,
        response_format={"type": "json_object"}
    )

    return response.choices[0].message.content


def safe_parse_json(text: str):
    import json

    try:
        data = json.loads(text)
        return {
            "annotated_code": data.get("annotated_code", ""),
            "generated_tests": data.get("generated_tests", ""),
            "notes": data.get("notes", ""),
        }
    except Exception:
        # fallback in case the model acts like a comedian instead of an engineer
        return {
            "annotated_code": "",
            "generated_tests": "",
            "notes": f"Could not parse model output as JSON.\n\nRaw output:\n{text}",
        }

# -----------------------------------------------------------------------------
# Main app function
# -----------------------------------------------------------------------------

def process_code(code, language, task, provider, model):
    if not code or not code.strip():
        return "", "", "Please paste some code first."

    try:
        raw = call_llm(code, language, task, provider, model)
        parsed = safe_parse_json(raw)

        annotated = parsed["annotated_code"]
        tests = parsed["generated_tests"]
        notes = parsed["notes"]

        if task == "Add docstrings/comments only":
            if not annotated:
                annotated = "No annotated code was returned."
        elif task == "Generate unit tests only":
            if not tests:
                tests = "No tests were returned."

        return annotated, tests, notes

    except Exception as e:
        return "", "", f"Error: {str(e)}"

# -----------------------------------------------------------------------------
# Example snippets
# -----------------------------------------------------------------------------

PYTHON_EXAMPLE = '''def divide(a, b):
    return a / b

class Calculator:
    def add(self, x, y):
        return x + y
'''

RUBY_EXAMPLE = '''def divide(a, b)
  a / b
end

class Calculator
  def add(x, y)
    x + y
  end
end
'''

def load_example(language):
    if language == "Python":
        return PYTHON_EXAMPLE
    return RUBY_EXAMPLE

def suggested_test_filename(language):
    return LANGUAGE_RULES[language]["default_test_filename"]

def update_models(provider):
    models = PROVIDER_MODELS.get(provider, [])
    return gr.Dropdown(choices=models, value=models[0])


### Gradio UI

In [ ]:
with gr.Blocks(title="Code Docstring/Comment + Test Generator") as demo:
    gr.Markdown("## Code Documentation & Unit Test Generator")
    gr.Markdown(
        "Paste code, choose the language, and generate docstrings/comments, tests, or both."
    )

    with gr.Row():
        language = gr.Dropdown(
            choices=["Python", "Ruby"],
            value="Python",
            label="Language",
        )
        task = gr.Dropdown(
            choices=[
                "Add docstrings/comments only",
                "Generate unit tests only",
                "Do both",
            ],
            value="Do both",
            label="Task",
        )
        provider = gr.Dropdown(
            choices=list(providers.keys()),
            value="OpenAI",
            label="Provider",
        )
        model = gr.Dropdown(
            choices=PROVIDER_MODELS[provider.value],
            value=PROVIDER_MODELS[provider.value][0],
            label="Model",
        )
        provider.change(
            fn=update_models,
            inputs=provider,
            outputs=model
        )

    with gr.Row():
        code_input = gr.Code(
            label="Input Code",
            language="python",
            value=PYTHON_EXAMPLE,
            lines=18,
        )
        annotated_output = gr.Code(
            label="Annotated Code",
            language="python",
            lines=20,
        )

        tests_output = gr.Code(
            label="Generated Tests",
            language="python",
            lines=20,
        )

    with gr.Row():
        load_btn = gr.Button("Load Example")
        run_btn = gr.Button("Generate", variant="primary")
        clear_btn = gr.Button("Clear")

    notes_output = gr.Textbox(
        label="Notes",
        lines=5,
    )

    test_filename = gr.Textbox(
        label="Suggested Test Filename",
        value="test_generated.py",
        interactive=False,
    )

    def on_language_change(lang):
        filename = suggested_test_filename(lang)
        example = load_example(lang)
        return (
            gr.update(value=example),
            filename,
        )

    language.change(
        fn=on_language_change,
        inputs=language,
        outputs=[code_input, test_filename],
    )

    load_btn.click(
        fn=lambda lang: load_example(lang),
        inputs=language,
        outputs=code_input,
    )

    run_btn.click(
        fn=process_code,
        inputs=[code_input, language, task, provider, model],
        outputs=[annotated_output, tests_output, notes_output],
    )

    clear_btn.click(
        fn=lambda: ("", "", ""),
        inputs=[],
        outputs=[code_input, annotated_output, notes_output],
    )

demo.launch()